# Análisis exploratorio: BAF Base

**Objetivo.** Conocer los datos antes de modelar y fijar las decisiones de
preprocesamiento con evidencia. Las conclusiones se trasladan a
`docs/datos.md` y a la memoria del TFM.

**Contenido**
1. Carga de los datos
2. Prevalencia de fraude por mes y por partición
3. Valores ausentes: ¿la ausencia se relaciona con el fraude?
4. Deriva entre meses
5. Atributo protegido (`customer_age >= 50`)

## 1. Carga de los datos

Se carga `Base.csv` con las rutas de `configs/local.yaml`. `resolve_path`
convierte la ruta en absoluta desde la raíz del repositorio, para que funcione
aunque el notebook se ejecute desde `notebooks/`.

In [ ]:
import pandas as pd

from src.config import load_config, resolve_path
from src.data import load_data, MISSING_AS_MINUS_ONE

config = load_config(resolve_path("configs/local.yaml"))
df = load_data(config)

df.shape


(1000000, 32)

**Resultado.** 1.000.000 de filas y 32 columnas, como indica la documentación
del BAF. La verificación de tipos, rangos y ausentes codificados como negativos
está en `docs/datos.md`.

## 2. Prevalencia de fraude por mes y por partición

Partición fija del proyecto: entrenamiento = meses 0–4, validación = mes 5,
test = meses 6–7.

**Qué se mira.** El número de solicitudes y la tasa de fraude en cada mes y en
cada partición.

**Por qué importa.** El umbral se fija en validación (5 % de FPR) y se aplica
tal cual al test. Si la distribución cambia entre el mes 5 y los meses 6–7, ese
umbral no dará exactamente un 5 % de FPR en test.

In [5]:
import numpy as np

""""
def particionar_df(x):
    if x <= 4:
        return "train"
    elif x == 5:
        return "eval"
    elif 5 < x <=7:
        return "test"
    else:
        return 0

df["split"] = df["month"].apply(particionar_df)


train = df[df["split"].str.contains("train")].copy().reset_index(drop=True)
eval = df[df["split"].str.contains("eval")].copy().reset_index(drop=True)
test = df[df["split"].str.contains("test")].copy().reset_index(drop=True)

print(len(train))

"""
conditions = [df["month"] <=4, df["month"] ==5, df["month"] >= 6]
df["split"] = np.select(conditions, ["train", "val", "test"], default = "unknown")

assert(df["split"] != "unknown").all(), "Hay meses fuera de 0-7"

df.groupby("split")["fraud_bool"].agg(["size", "sum", "mean"])



,size,sum,mean
split,,,
test,205011,2878,0.014038
train,675666,6740,0.009975
val,119323,1411,0.011825


In [4]:
df.groupby("month")["fraud_bool"].agg(["size", "sum", "mean"])

,size,sum,mean
month,,,
0,132440,1500,0.011326
1,127620,1198,0.009387
2,136979,1198,0.008746
3,150936,1392,0.009222
4,127691,1452,0.011371
5,119323,1411,0.011825
6,108168,1450,0.013405
7,96843,1428,0.014746


## 3. Valores ausentes: ¿la ausencia se relaciona con el fraude?

**Qué se mira.**
- En las columnas con -1 como ausente, la tasa de fraude con y sin el dato.
- En `intended_balcon_amount`, la tasa de fraude por tramos de los valores
  negativos, para decidir si son ausentes (decisión pendiente en
  `docs/datos.md`).

**Por qué importa.** Si la ausencia predice el fraude, el indicador de
ausencia que usarán las redes aporta información.

In [12]:
df_minus_one = df[MISSING_AS_MINUS_ONE]

assert(df_minus_one >= -1).all().all(), "Hay valores distintos de -1"

In [15]:
rows = []

for col in MISSING_AS_MINUS_ONE:
    is_missing = (df[col] == -1)
    rows.append({
        "column": col,
        "n_missing": is_missing.sum(),
        "pct_missing": is_missing.mean(),
        "fraud_rate_missing": df.loc[is_missing, "fraud_bool"].mean(),
        "fraud_rate_present": df.loc[~is_missing, "fraud_bool"].mean(),
        "ratio": df.loc[is_missing, "fraud_bool"].mean() / df.loc[~is_missing, "fraud_bool"].mean()
    })


summary = pd.DataFrame(rows)
summary

,column,n_missing,pct_missing,fraud_rate_missing,fraud_rate_present,ratio
0,prev_address_months_count,712920,0.712920,0.014215,0.003118,4.559529
1,current_address_months_count,4254,0.004254,0.003291,0.011062,0.297505
2,bank_months_count,253635,0.253635,0.016323,0.009230,1.768423
3,session_length_in_minutes,2015,0.002015,0.008933,0.011033,0.809645
4,device_distinct_emails_8w,359,0.000359,0.011142,0.011029,1.010255


## 4. Deriva entre meses

**Qué se mira.** Si la distribución de las variables cambia de un mes a otro,
sobre todo entre entrenamiento (0–4) y test (6–7).

**Por qué importa.** El modelo se entrena con el pasado y se evalúa con el
futuro. Una deriva fuerte explicaría caídas de rendimiento en test.

In [ ]:
# Variables de entrada numéricas: sin el objetivo, el mes y la constante
num_cols = df.select_dtypes(include="number").columns.drop(
    ["fraud_bool", "month", "device_fraud_count"]
)

df.groupby("month")[num_cols].mean()

In [ ]:
# Proporción de cada categoría por mes (cada fila suma 1)
cat_cols = df.select_dtypes(exclude="number").columns.drop("split")

for col in cat_cols:
    display(pd.crosstab(df["month"], df[col], normalize="index").round(3))

In [ ]:
def psi(reference, actual, n_bins=10, max_discrete=20, eps=1e-6):
    """Population Stability Index de `actual` respecto a `reference`.

    - Variables con pocos valores distintos (binarias, categóricas, discretas):
      un tramo por valor.
    - Variables continuas: tramos por cuantiles de la referencia.
    """
    reference = pd.Series(reference)
    actual = pd.Series(actual)

    if not pd.api.types.is_numeric_dtype(reference) or reference.nunique() <= max_discrete:
        # Un tramo por valor; se incluyen los valores que solo aparecen en `actual`
        values = reference.value_counts().index.union(actual.value_counts().index)
        p_ref = reference.value_counts(normalize=True).reindex(values, fill_value=0).to_numpy()
        p_act = actual.value_counts(normalize=True).reindex(values, fill_value=0).to_numpy()
    else:
        # Bordes de los tramos: cuantiles de la referencia, con extremos abiertos
        edges = np.unique(np.quantile(reference, np.linspace(0, 1, n_bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        p_ref = np.histogram(reference, bins=edges)[0] / len(reference)
        p_act = np.histogram(actual, bins=edges)[0] / len(actual)

    # Evitar divisiones por 0 y log(0)
    p_ref = np.clip(p_ref, eps, None)
    p_act = np.clip(p_act, eps, None)

    return np.sum((p_act - p_ref) * np.log(p_act / p_ref))

In [ ]:
# Comprobación de psi() con casos de resultado conocido
rng = np.random.default_rng(0)
a = rng.normal(0, 1, 100_000)

print("Continua, misma distribución: ", round(psi(a, rng.normal(0, 1, 100_000)), 4))    # ≈ 0
print("Continua, desplazada 0,5:     ", round(psi(a, rng.normal(0.5, 1, 100_000)), 4))  # > 0,1
print("Binaria, misma proporción:    ", round(psi(rng.binomial(1, 0.3, 100_000), rng.binomial(1, 0.3, 100_000)), 4))  # ≈ 0
print("Binaria, de 0,3 a 0,5:        ", round(psi(rng.binomial(1, 0.3, 100_000), rng.binomial(1, 0.5, 100_000)), 4))  # > 0,1

In [ ]:
# PSI de test (meses 6–7) respecto a train (meses 0–4), numéricas y categóricas
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

rows = []
for col in [*num_cols, *cat_cols]:
    rows.append({"column": col, "psi": psi(train[col], test[col])})

psi_table = pd.DataFrame(rows).sort_values("psi", ascending=False).round(4)
psi_table

## 5. Atributo protegido: edad

Grupos: `customer_age >= 50` frente a `< 50`, el corte del código oficial del
BAF (ver `docs/datos.md`).

**Qué se mira.** El tamaño de cada grupo y su tasa de fraude, en total y por
partición.

**Por qué importa.** La métrica de equidad es el cociente de FPR entre los
dos grupos. Un grupo pequeño en test da un FPR con mucha varianza.